In [5]:
from google_sheet_import import load_processed_data
import pandas as pd
import numpy as np
import matplotlib

import faicons as fa
import plotly.express as px

# Load data and compute static values
from shinywidgets import render_plotly

from shiny import reactive, render
from shiny.express import input, ui, app

# importing data
df_full, tranny = load_processed_data()


In [6]:
df_full[df_full['Category']=='Accommodation'][df_full['Person']=='Jesse'].head(10)
# daily_total = df_full.groupby("Expense Date")["Price NZD"].sum()
# daily_total.mean()


C:\Users\jessep.LAPTOP-7GPOPVDF\AppData\Local\Temp\ipykernel_44116\3509401250.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_full[df_full['Category']=='Accommodation'][df_full['Person']=='Jesse'].head(10)


,Timestamp,Category,Price,Currency,Person,First Night in Accom,Total Nights in Accom,Expense Date,Extra Note,Price NZD
2,2026-03-29 17:40:00,Accommodation,110.0,EUR,Jesse,2026-03-29,1,2026-03-29,Hotel Lisbon pre-Camino,220.206995
3,2026-03-29 17:40:00,Accommodation,110.0,EUR,Jesse,2026-03-29,1,2026-03-30,Hotel Lisbon pre-Camino,220.746122
12,2026-04-01 18:30:00,Accommodation,25.0,EUR,Jesse,2026-04-01,1,2026-04-01,Albergue,50.255297
16,2026-04-02 19:00:00,Accommodation,18.0,EUR,Jesse,2026-04-02,1,2026-04-02,Albergue,36.343812
20,2026-04-03 19:20:00,Accommodation,22.0,EUR,Jesse,2026-04-03,1,2026-04-03,Hostel private room,44.420215
24,2026-04-04 18:50:00,Accommodation,15.0,EUR,Jesse,2026-04-04,1,2026-04-04,Donativo,30.286510
28,2026-04-05 19:10:00,Accommodation,30.0,EUR,Jesse,2026-04-05,1,2026-04-05,Guesthouse,60.573021
32,2026-04-06 19:00:00,Accommodation,20.0,EUR,Jesse,2026-04-06,1,2026-04-06,Albergue,40.382014
36,2026-04-07 18:40:00,Accommodation,17.0,EUR,Jesse,2026-04-07,1,2026-04-07,Albergue,34.454105
40,2026-04-08 19:30:00,Accommodation,35.0,EUR,Jesse,2026-04-08,1,2026-04-08,Hotel (rest day),70.192327


In [7]:
# Templates configuration
# -----------------------
#     Default template: 'plotly'
#     Available templates:
#         ['ggplot2', 'seaborn', 'simple_white', 'plotly',
#          'plotly_white', 'plotly_dark', 'presentation', 'xgridoff',
#          'ygridoff', 'gridon', 'none']
import plotly.express as px
for template in  ['ggplot2', 'seaborn', 'simple_white', 'plotly',
         'plotly_white', 'plotly_dark', 'presentation', 'xgridoff',
         'ygridoff', 'gridon', 'none']:

    df = df_full
    df = df[df["Category"] != "Flight"].copy()

    df["Expense Date"] = pd.to_datetime(df["Expense Date"]).dt.normalize()


    start_date = df["Expense Date"].min()
    end_date = df["Expense Date"].max()
    all_dates = pd.date_range(start=start_date, end=end_date, freq="D")

    all_categories = df["Category"].dropna().unique()

    full_index = pd.MultiIndex.from_product(
        [all_dates, all_categories],
        names=["Expense Date", "Category"]
    )

    daily_category = (
        df.groupby(["Expense Date", "Category"])["Price NZD"]
        .sum()
        .reindex(full_index, fill_value=0)
        .reset_index()
    )

    daily_total = (
                    daily_category.groupby("Expense Date", as_index=False)["Price NZD"]
                    .sum()
                )
    daily_total["Category"] = "Total"

    # --- combine with existing data ---
    plot_df = pd.concat([daily_category, daily_total], ignore_index=True)

    plot_df["Date"] = plot_df["Expense Date"].dt.strftime("%Y-%m-%d")

    fig = px.line(
        plot_df,
        x="Date",
        y="Price NZD",
        color="Category",
        markers=True,
        title="Daily Spend by Category",
        hover_data=["Price NZD", "Category"],
        template=template
    )

    fig.update_xaxes(type="date", tickformat="%d %b")

    fig.update_layout(
        xaxis_title="Date",
        yaxis_title="Spend NZD",
        hovermode="x unified"
    )

    fig.update_traces(
        hovertemplate=
        "<b>%{fullData.name}</b><br>" +   # Category
        "Spend: $%{y:.2f}<extra></extra>"  # Value
    )

    fig.show()

In [8]:
df_full[df_full['Person'] == 'Jesse']["Price NZD"].sum()

np.float64(6461.00062726119)

In [10]:
df_full['Category'].unique()

array(['Flight', 'Accommodation', 'Food', 'Misc', 'Transport'],
      dtype=object)

In [74]:
import pandas as pd
import numpy as np
import os
from googleapiclient.discovery import build
from google.oauth2.service_account import Credentials
import requests


# Configuration
SERVICE_ACCOUNT_FILE = 'camino_key.json'
SPREADSHEET_ID = '1MVeRNsn2NJaLaHRGiSYtZYq9RyKD7MNE7bYYAMbUX6g'
                    
RANGE_NAME = 'Locations' 

#Gets google sheet data as a list 
scopes = ['https://www.googleapis.com/auth/spreadsheets.readonly']
creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=scopes)

# Build the service
service = build('sheets', 'v4', credentials=creds)

# Call the Sheets API
sheet = service.spreadsheets()
result = sheet.values().get(spreadsheetId=SPREADSHEET_ID, range=RANGE_NAME).execute()
values = result.get('values', [])

#Converting to dataframe
headers = values[0]
rows = values[1:]



df = pd.DataFrame(rows, columns=headers)

df["Date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
df["latitude"] = df["latitude"].astype("float")
df["longitude"] = df["longitude"].astype("float")
df["town"] = df["town"].astype("string")

In [76]:
df.head(10)

,Date,latitude,longitude,town
0,2026-04-05,38.72509,-9.14980,Lisbon
1,2026-04-06,38.84629,-9.08748,Santa Iria de Azoia
2,2026-04-07,38.95525,-8.98966,Vila Franca de Xira
3,2026-04-08,39.07029,-8.86822,Azambuja
4,2026-04-09,39.23379,-8.68617,Santarem
5,2026-04-10,39.40270,-8.48908,Golega
6,2026-04-11,39.60199,-8.40924,Tomar
7,2026-04-12,39.81951,-8.38858,Alvaiazere
8,2026-04-13,39.93424,-8.42045,Ansiao
9,2026-04-14,40.11573,-8.49834,Condeixa-a-Nova


In [86]:
# df_full.head()
daily_category = (
        df_full.groupby(["Expense Date", "Category"])["Price NZD"]
        .sum()
        .reindex(full_index, fill_value=0)
        .reset_index()
    )

daily_total = (
                daily_category.groupby("Expense Date", as_index=False)["Price NZD"]
                .sum()
            )
daily_total["Category"] = "Total"

map_df = pd.concat([daily_category, daily_total], ignore_index=True)
map_df =  map_df.pivot(index = 'Expense Date', columns = 'Category', values = 'Price NZD')
map_df =  pd.merge(map_df, df, left_on='Expense Date', right_on = 'Date', how='inner')
map_df["Date"] = pd.to_datetime(map_df["Date"]).dt.strftime("%Y-%m-%d")


In [88]:
map_df

,Accommodation,Food,Misc,Total,Transport,Date,latitude,longitude,town
0,121.146042,100.955035,0.000000,222.101076,0.0,2026-04-05,38.72509,-9.14980,Lisbon
1,80.764028,36.343812,18.171906,135.279746,0.0,2026-04-06,38.84629,-9.08748,Santa Iria de Azoia
2,68.908210,70.934922,0.000000,139.843132,0.0,2026-04-07,38.95525,-8.98966,Vila Franca de Xira
3,140.384654,54.148367,28.076931,222.609951,0.0,2026-04-08,39.07029,-8.86822,Azambuja
4,60.056453,62.058335,0.000000,122.114788,0.0,2026-04-09,39.23379,-8.68617,Santarem
5,72.122608,38.064710,10.017029,120.204347,0.0,2026-04-10,39.40270,-8.48908,Golega
6,80.136232,78.132826,0.000000,158.269057,0.0,2026-04-11,39.60199,-8.40924,Tomar
7,88.149855,86.146449,0.000000,174.296304,0.0,2026-04-12,39.81951,-8.38858,Alvaiazere
8,100.405639,42.170368,36.146030,178.722037,0.0,2026-04-13,39.93424,-8.42045,Ansiao
9,67.972811,69.972011,0.000000,137.944822,0.0,2026-04-14,40.11573,-8.49834,Condeixa-a-Nova


In [112]:
import plotly.graph_objs as go
import numpy as np

lat = map_df["latitude"].astype(float)
lon = map_df["longitude"].astype(float)

# Identify category columns (everything except non-category fields)
exclude_cols = {"Expense Date", "Date", "latitude", "longitude", "town"}
category_cols = [c for c in map_df.columns if c not in exclude_cols]

# Build customdata: town, date, then all category values
customdata = np.stack(
    [map_df["town"], map_df["Date"]] + [map_df[c] for c in category_cols],
    axis=-1
)

# Build hovertemplate dynamically
hover_lines = [
    "<b>%{customdata[0]}</b>",
    "Date: %{customdata[1]}",
]

for i, col in enumerate(category_cols, start=2):
    # if map_df[col][i] != 0:
    hover_lines.append(f"{col}: $%{{customdata[{i}]:.2f}}")

hovertemplate = "<br>".join(hover_lines) + "<extra></extra>"

# ---- Plot ----
fig = go.Figure(
    go.Scattermap(
        lat=lat,
        lon=lon,
        mode="markers+lines",
        customdata=customdata,
        hovertemplate=hovertemplate,
        hoverlabel=dict(namelength=-1),
    )
)

# Keep your zoom logic
padding = 0.2
center = {"lat": float(lat.mean()), "lon": float(lon.mean())}

lat_range = (lat.max() - lat.min()) + padding * 2
lon_range = (lon.max() - lon.min()) + padding * 2
max_range = max(lat_range, lon_range)

zoom = 8 - np.log(max_range + 1e-6)
zoom = max(min(zoom, 15), 3)

fig.update_layout(
    map=dict(center=center, zoom=zoom),
    margin={"r": 0, "t": 0, "l": 0, "b": 0},
)

fig.show()